# recipe-dataclass — faded example 2: Recipe parents for a binary sub_forward

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`. The last cell reports your progress on the `Backprop: Recipe dataclass` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A binary forward records both Tensor inputs in `parents`, keyed by positional argnum: `{0: a, 1: b}`. Each value is the original `MiniTensor` object, because the reverse pass accumulates `.grad` onto those exact objects. Omitting a parent orphans that branch of the graph.

## Faded exercise 2

Complete `sub_forward(a, b)` for `MiniTensor` inputs computing `out = a.array - b.array`. The `func`, `args`, and `kwargs` of the `Recipe` are already filled. Fill in the `parents` dict so both Tensor inputs are recorded by their positional argnum.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def sub_forward(a: MiniTensor, b: MiniTensor) -> MiniTensor:
    out = MiniTensor(a.array - b.array)
    parents = None  # TODO: fill in this step — read the prompt cell above
    out.recipe = Recipe(
        func=t.sub,
        args=(a.array, b.array),
        kwargs={},
        parents=parents,
    )
    return out


a = MiniTensor(t.tensor([5.0, 7.0]))
b = MiniTensor(t.tensor([2.0, 1.0]))
out = sub_forward(a, b)

def _test():
    a = MiniTensor(t.tensor([5.0, 7.0]))
    b = MiniTensor(t.tensor([2.0, 1.0]))
    out = sub_forward(a, b)
    r = out.recipe
    assert set(r.parents.keys()) == {0, 1}, 'parents must key argnums 0 and 1'
    assert r.parents[0] is a, 'parent 0 must be a'
    assert r.parents[1] is b, 'parent 1 must be b'
    assert t.allclose(out.array, t.tensor([3.0, 6.0])), 'forward value must be a - b'

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def sub_forward(a: MiniTensor, b: MiniTensor) -> MiniTensor:
    out = MiniTensor(a.array - b.array)
    parents = {0: a, 1: b}
    out.recipe = Recipe(
        func=t.sub,
        args=(a.array, b.array),
        kwargs={},
        parents=parents,
    )
    return out


a = MiniTensor(t.tensor([5.0, 7.0]))
b = MiniTensor(t.tensor([2.0, 1.0]))
out = sub_forward(a, b)
```
</details>